## Evaluation pipeline for the microlane experiment

In [ ]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [ ]:
# Imports of the Core Packages
import json, sys, time, pytz
import os, yaml,random 
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Import custom libraries located at different folder location + configs
from microlane.utils.metrics import *
from microlane.datasets.tusimple import TuSimple
from microlane.datasets.raw import Raw
from microlane.models.lanenet2.model import LaneNet2
from microlane.models.ufld.model import UFLD
from microlane.schema.output import ModelPrediction
from microlane.schema.sample import Sample
from microlane.utils.load_image import load_image_from_sample
from microlane.utils.experiment import ExperimentEvaluate

In [ ]:
# First Load the Configuation file
with open("../configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

### Pre Processing Part

In [ ]:
# First initialise the dataset
# Then load the dataset
dataset = Raw(
    
    folder_path="/mnt/c/Users/suyog/Downloads/AI Images"
)

data = dataset.load_raw(number=10)

In [ ]:
# So, basically now we will import the model
# model = LaneNet2() type and what we will do is, run 
# Run model.inference(formatted_dataset)
# Ensure that Docker Engine Is Running in background

model = LaneNet2(
    
    container_folder=config['models']['lanenet2']['container_folder'],
    
    image_name=config['models']['lanenet2']['image_name']
)

In [ ]:
experiment = ExperimentEvaluate(
    
    experiment_name="pipeline testing with no augmentation",
    
    output_dir="/home/suyog/desktop/projects/microlane/results/testing"
    
)

### Models and Datasets Loaded, Now Processing Part

In [ ]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

random_image_index = random.randint(0, len(data)-1)

item = data[random_image_index]
print(f"Index        : {random_image_index}")
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

### Adding Image Augmentation

In [ ]:
blur_range = tuple(config['data']['augmentation']['blur'])
rotation_range = tuple(config['data']['augmentation']['rotation'])
zoom_range = tuple(config['data']['augmentation']['zoom'])
lighting_range =tuple(config['data']['augmentation']['lighting'])
motion_blur_range=tuple(config['data']['augmentation']['motion_blur'])

print(f"blur_range:        {blur_range}")
print(f"rotation_range:    {rotation_range}")
print(f"zoom_range:        {zoom_range}")
print(f"lighting_range:    {lighting_range}")
print(f"motion_blur_range: {motion_blur_range}")

In [ ]:
PRESETS = {
    "normal":       dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.0, motion_blur=0.0, shake=False),
    "lighting_d":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=-0.4,  motion_blur=0.0, shake=False), 
    "lighting_b":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.4,  motion_blur=0.0, shake=False),
    "motion-blur":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.0,  motion_blur=1, shake=False),
    "camera-shake": dict(blur=0, rotation=0.0, zoom=1.0, lighting=0.0,  motion_blur=0, shake=True),
}

EFFECT = "normal"
AUGMENTATION_SETTINGS = PRESETS[EFFECT]

item.blur        = AUGMENTATION_SETTINGS["blur"]
item.rotation    = AUGMENTATION_SETTINGS["rotation"]
item.zoom        = AUGMENTATION_SETTINGS["zoom"]
item.lighting    = AUGMENTATION_SETTINGS["lighting"]
item.motion_blur = AUGMENTATION_SETTINGS["motion_blur"]

if AUGMENTATION_SETTINGS['shake']:
    
    item.rotation = random.uniform(-10.0, 10.0)   # degrees
    
    item.zoom     = random.uniform(1, 1.15)  # subtle scale jitter


print(f"Effect:      {EFFECT}")
print(f"Blur:        {item.blur:.2f}")
print(f"Rotation:    {item.rotation:.2f}")
print(f"Zoom:        {item.zoom:.2f}")
print(f"Lighting:    {item.lighting:.2f}")
print(f"Motion blur: {item.motion_blur:.2f}")
print(f"Shake:       {AUGMENTATION_SETTINGS['shake']}")

In [ ]:

loaded_image = load_image_from_sample(item)


In [ ]:
if loaded_image.image is not None:
    plt.imshow(loaded_image.image)
    plt.axis("off")
    plt.title("Augmented Image")
    plt.show()

In [ ]:
# We are basically sending a loaded sample with actual image tensor in the memory

response = model.predict(loaded_image)

In [ ]:
prediction = response.json()

ModelOutput = ModelPrediction (
    
    sample=Sample(
            image=prediction['sample']['image'],
            image_path=prediction['sample']['image_path'],
            h_samples=prediction['sample']['h_samples'],
            lanes=prediction['sample']['lanes'],
            blur=prediction['sample']['blur'],
            lighting=prediction['sample']['lighting'],
            zoom=prediction['sample']['zoom'],
            rotation=prediction['sample']['rotation']
        ),
    
    lanes=prediction['lanes'],
        
    run_time=prediction['run_time']
    
)

In [ ]:

experiment.store_prediction(ModelOutput)

In [ ]:
experiment.visualize_prediction(ModelOutput, show=True)